# Robustness Analysis

This notebook tests how sensitive the portfolio results are to key modeling choices.

The robustness analysis varies:

- estimation window
- rebalancing frequency
- volatility target
- maximum leverage

Estimation window and rebalancing frequency are evaluated on raw portfolios without volatility targeting.

Volatility-target and leverage sensitivity are evaluated around the base-case implementation of a 10% volatility target and a 2x leverage cap.

In [12]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd

import metrics
import backtesting

importlib.reload(metrics)
importlib.reload(backtesting)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01
VOL_WINDOW = 60

## Data and strategies

The same ETF universe, risk-free rate and portfolio strategies used in the main backtest are reused here.

Each robustness test changes one modeling assumption while keeping the others fixed.

In [2]:
prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

In [13]:
def run_robustness_returns(window=WINDOW, step=STEP, vol_target=VOL_TARGET,
                           max_leverage=MAX_LEVERAGE,
                           transaction_cost=TRANSACTION_COST,
                           funding_spread=FUNDING_SPREAD, vol_window=VOL_WINDOW):

    portfolio_returns = {}

    for name, (strategy, strategy_groups) in strategies.items():
        port_ret, _, _, _, _ = backtesting.backtest(
            returns,
            window=window,
            step=step,
            allocation_func=strategy,
            groups=strategy_groups,
            vol_target=vol_target,
            max_leverage=max_leverage,
            risk_free_returns=risk_free_returns,
            transaction_cost=transaction_cost,
            funding_spread=funding_spread,
            vol_window=vol_window
        )

        portfolio_returns[name] = port_ret

    return pd.DataFrame(portfolio_returns)


def performance_table(port_returns):
    results = []

    for strategy in port_returns.columns:
        port_ret = port_returns[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    return pd.DataFrame(results).set_index("Strategy")

## Estimation window

The estimation window controls how much historical data is used to estimate portfolio inputs.

The comparison uses **2-year, 3-year and 5-year** rolling windows on **raw portfolios without volatility targeting**.

Because different window lengths produce different out-of-sample start dates, all results are aligned to a common evaluation period before computing performance metrics.

In [4]:
windows = {
    "2Y": 2 * 252,
    "3Y": 3 * 252,
    "5Y": 5 * 252
}

window_returns = {}

for label, window in windows.items():
    window_returns[label] = run_robustness_returns(window=window, vol_target=None)

common_index = (
    window_returns["2Y"].index
    .intersection(window_returns["3Y"].index)
    .intersection(window_returns["5Y"].index)
)

for label, port_returns in window_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Estimation Window — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.01%,4.33%,0.583,-12.14%
Max Geo,10.69%,15.83%,0.627,-32.42%
Max Sharpe,5.60%,10.31%,0.435,-26.50%
Carver,3.19%,3.11%,0.541,-10.36%
Hierarchical Inv Vol,3.37%,3.36%,0.557,-10.65%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.15%,4.27%,0.622,-11.70%
Max Geo,13.70%,15.45%,0.811,-27.77%
Max Sharpe,7.19%,9.52%,0.619,-24.99%
Carver,3.26%,3.08%,0.568,-10.07%
Hierarchical Inv Vol,3.45%,3.32%,0.585,-10.25%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.10%,4.17%,0.624,-11.80%
Max Geo,12.04%,15.93%,0.699,-30.41%
Max Sharpe,6.72%,9.02%,0.600,-26.02%
Carver,3.18%,3.01%,0.555,-10.16%
Hierarchical Inv Vol,3.36%,3.24%,0.571,-10.37%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


## Rebalancing frequency

The rebalancing interval is varied between:

- weekly
- monthly
- quarterly

This test is performed on **raw portfolios without volatility targeting** to isolate the effect of rebalancing frequency.

In [5]:
steps = {
    "Weekly": 5,
    "Monthly": 21,
    "Quarterly": 63
}

step_returns = {}

for label, step in steps.items():
    step_returns[label] = run_robustness_returns(step=step, vol_target=None)

common_index = (
    step_returns["Weekly"].index
    .intersection(step_returns["Monthly"].index)
    .intersection(step_returns["Quarterly"].index)
)

for label, port_returns in step_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Rebalancing Frequency — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.40%,14.87%,0.459,-41.49%
Inverse Vol,3.78%,5.05%,0.478,-15.70%
Max Geo,10.86%,15.48%,0.652,-37.25%
Max Sharpe,5.80%,8.97%,0.516,-26.09%
Carver,3.18%,3.42%,0.518,-10.14%
Hierarchical Inv Vol,3.34%,3.74%,0.517,-10.32%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.31%,11.34%,0.635,-31.47%
Tech,16.43%,22.41%,0.728,-49.37%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.31%,14.60%,0.459,-41.61%
Inverse Vol,3.77%,4.94%,0.486,-15.47%
Max Geo,11.28%,16.22%,0.653,-39.34%
Max Sharpe,6.56%,9.06%,0.590,-24.99%
Carver,3.17%,3.38%,0.521,-10.07%
Hierarchical Inv Vol,3.33%,3.69%,0.521,-10.25%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.28%,11.11%,0.643,-31.48%
Tech,16.43%,22.41%,0.728,-49.37%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.27%,14.32%,0.462,-41.42%
Inverse Vol,3.75%,4.89%,0.486,-15.42%
Max Geo,11.13%,16.59%,0.634,-39.69%
Max Sharpe,6.34%,9.38%,0.551,-22.69%
Carver,3.14%,3.35%,0.516,-10.11%
Hierarchical Inv Vol,3.29%,3.66%,0.516,-10.29%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.32%,10.93%,0.656,-31.20%
Tech,16.43%,22.41%,0.728,-49.37%


## Volatility target

The annual volatility target is varied between **5%, 10% and 15%**.

The leverage cap remains fixed at 2x, so some low-volatility strategies may be unable to reach the higher targets.

In [14]:
vol_targets = {
    "5%": 0.05,
    "10%": 0.10,
    "15%": 0.15
}

vol_target_returns = {}

for label, vol_target in vol_targets.items():
    vol_target_returns[label] = run_robustness_returns(vol_target=vol_target, vol_window=VOL_WINDOW)

common_index = (
    vol_target_returns["5%"].index
    .intersection(vol_target_returns["10%"].index)
    .intersection(vol_target_returns["15%"].index)
)

for label, port_returns in vol_target_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Volatility Target — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,3.93%,5.63%,0.461,-11.80%
Inverse Vol,3.53%,5.32%,0.412,-14.58%
Max Geo,5.58%,5.83%,0.716,-11.76%
Max Sharpe,4.68%,5.00%,0.655,-14.79%
Carver,3.33%,4.81%,0.410,-15.07%
Hierarchical Inv Vol,3.51%,4.94%,0.434,-14.87%
SPY,5.00%,5.85%,0.620,-11.13%
60/40,5.40%,5.85%,0.686,-10.85%
Tech,5.73%,5.65%,0.763,-9.06%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,6.03%,11.28%,0.450,-22.81%
Inverse Vol,4.29%,8.43%,0.372,-24.24%
Max Geo,9.57%,11.73%,0.716,-22.92%
Max Sharpe,6.69%,8.27%,0.653,-22.99%
Carver,3.61%,6.45%,0.362,-20.97%
Hierarchical Inv Vol,3.81%,6.91%,0.370,-21.27%
SPY,8.37%,11.82%,0.619,-21.63%
60/40,8.70%,11.45%,0.661,-20.79%
Tech,9.92%,11.36%,0.765,-18.05%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.50%,16.28%,0.439,-31.60%
Inverse Vol,4.70%,9.36%,0.385,-29.09%
Max Geo,12.92%,17.61%,0.698,-32.93%
Max Sharpe,8.09%,10.98%,0.634,-29.13%
Carver,3.78%,6.73%,0.374,-21.25%
Hierarchical Inv Vol,4.08%,7.28%,0.391,-21.59%
SPY,11.22%,17.65%,0.611,-31.46%
60/40,11.19%,15.52%,0.670,-29.61%
Tech,13.81%,17.13%,0.758,-26.36%


## Maximum leverage

The maximum allowed portfolio leverage is varied between **1x, 1.5x and 2x**.

The volatility target remains fixed at 10%.

In [15]:
leverage_caps = {
    "1x": 1.0,
    "1.5x": 1.5,
    "2x": 2.0
}

leverage_returns = {}

for label, max_leverage in leverage_caps.items():
    leverage_returns[label] = run_robustness_returns(max_leverage=max_leverage, vol_window=VOL_WINDOW)

common_index = (
    leverage_returns["1x"].index
    .intersection(leverage_returns["1.5x"].index)
    .intersection(leverage_returns["2x"].index)
)

for label, port_returns in leverage_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Maximum Leverage — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.64%,9.91%,0.459,-21.84%
Inverse Vol,3.78%,4.79%,0.502,-15.30%
Max Geo,8.76%,11.15%,0.682,-22.92%
Max Sharpe,5.87%,6.72%,0.671,-17.61%
Carver,3.17%,3.38%,0.521,-10.07%
Hierarchical Inv Vol,3.33%,3.68%,0.522,-10.25%
SPY,8.09%,11.25%,0.622,-21.18%
60/40,7.52%,8.85%,0.703,-20.26%
Tech,9.70%,11.29%,0.751,-18.05%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.95%,11.00%,0.451,-21.84%
Inverse Vol,4.13%,6.81%,0.419,-20.92%
Max Geo,9.50%,11.71%,0.712,-22.92%
Max Sharpe,6.34%,7.56%,0.663,-20.34%
Carver,3.49%,5.00%,0.427,-15.81%
Hierarchical Inv Vol,3.71%,5.40%,0.437,-16.08%
SPY,8.29%,11.76%,0.616,-21.63%
60/40,8.52%,10.73%,0.683,-20.31%
Tech,9.91%,11.36%,0.764,-18.05%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,6.03%,11.28%,0.450,-22.81%
Inverse Vol,4.29%,8.43%,0.372,-24.24%
Max Geo,9.57%,11.73%,0.716,-22.92%
Max Sharpe,6.69%,8.27%,0.653,-22.99%
Carver,3.61%,6.45%,0.362,-20.97%
Hierarchical Inv Vol,3.81%,6.91%,0.370,-21.27%
SPY,8.37%,11.82%,0.619,-21.63%
60/40,8.70%,11.45%,0.661,-20.79%
Tech,9.92%,11.36%,0.765,-18.05%


## Volatility Estimation Window

This section tests the sensitivity of volatility-targeted portfolios to the horizon used to estimate portfolio risk.

The allocation estimation window remains fixed at 3 years, while the volatility estimation window varies from 20 to 756 trading days.

The volatility target is fixed at 10% and maximum leverage at 2x.

In [16]:
vol_windows = {
    "20D": 20,
    "60D": 60,
    "126D": 126,
    "252D": 252,
    "756D": 756
}

vol_window_returns = {}

for label, vol_window in vol_windows.items():
    vol_window_returns[label] = run_robustness_returns(
        vol_target=VOL_TARGET,
        vol_window=vol_window
    )

common_index = (
    vol_window_returns["20D"].index
    .intersection(vol_window_returns["60D"].index)
    .intersection(vol_window_returns["126D"].index)
    .intersection(vol_window_returns["252D"].index)
    .intersection(vol_window_returns["756D"].index)
)

for label, port_returns in vol_window_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Volatility Estimation Window — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,6.08%,11.62%,0.444,-24.32%
Inverse Vol,4.53%,8.09%,0.412,-24.22%
Max Geo,10.36%,12.07%,0.759,-20.87%
Max Sharpe,6.90%,8.26%,0.677,-23.10%
Carver,3.50%,6.40%,0.347,-21.23%
Hierarchical Inv Vol,3.84%,6.80%,0.379,-21.58%
SPY,8.88%,12.52%,0.629,-22.76%
60/40,9.10%,11.52%,0.690,-24.03%
Tech,11.29%,11.87%,0.842,-20.85%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,6.03%,11.28%,0.450,-22.81%
Inverse Vol,4.29%,8.43%,0.372,-24.24%
Max Geo,9.57%,11.73%,0.716,-22.92%
Max Sharpe,6.69%,8.27%,0.653,-22.99%
Carver,3.61%,6.45%,0.362,-20.97%
Hierarchical Inv Vol,3.81%,6.91%,0.370,-21.27%
SPY,8.37%,11.82%,0.619,-21.63%
60/40,8.70%,11.45%,0.661,-20.79%
Tech,9.92%,11.36%,0.765,-18.05%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.50%,11.26%,0.406,-24.64%
Inverse Vol,4.06%,8.59%,0.340,-26.14%
Max Geo,8.99%,11.41%,0.687,-22.39%
Max Sharpe,6.50%,8.25%,0.632,-22.86%
Carver,3.66%,6.52%,0.366,-21.19%
Hierarchical Inv Vol,3.85%,7.00%,0.371,-21.47%
SPY,7.80%,11.46%,0.589,-21.38%
60/40,8.03%,11.36%,0.611,-21.34%
Tech,9.17%,11.03%,0.722,-18.75%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.06%,11.17%,0.370,-26.09%
Inverse Vol,4.14%,8.70%,0.346,-26.50%
Max Geo,8.61%,11.12%,0.670,-22.68%
Max Sharpe,6.63%,8.30%,0.644,-24.81%
Carver,3.74%,6.64%,0.372,-21.25%
Hierarchical Inv Vol,3.95%,7.16%,0.378,-21.59%
SPY,7.24%,11.31%,0.549,-23.16%
60/40,7.75%,11.27%,0.592,-23.46%
Tech,8.89%,10.91%,0.705,-20.39%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.23%,10.86%,0.393,-30.50%
Inverse Vol,4.06%,9.01%,0.329,-29.73%
Max Geo,8.03%,10.57%,0.650,-24.03%
Max Sharpe,6.72%,8.40%,0.648,-22.58%
Carver,3.76%,6.73%,0.371,-21.25%
Hierarchical Inv Vol,4.00%,7.30%,0.379,-21.59%
SPY,6.63%,11.06%,0.507,-30.91%
60/40,7.41%,11.11%,0.571,-31.05%
Tech,8.47%,10.64%,0.683,-26.99%


In [19]:
# Save robustness results

def build_robustness_table(results_dict):
    common_index = None

    for port_returns in results_dict.values():
        common_index = (
            port_returns.index if common_index is None
            else common_index.intersection(port_returns.index)
        )

    tables = []

    for scenario, port_returns in results_dict.items():
        table = performance_table(port_returns.loc[common_index]).reset_index()
        table.insert(0, "Scenario", scenario)
        tables.append(table)

    return pd.concat(tables, ignore_index=True)


window_results = build_robustness_table(window_returns)
vol_window_results = build_robustness_table(vol_window_returns)
step_results = build_robustness_table(step_returns)
vol_target_results = build_robustness_table(vol_target_returns)
leverage_results = build_robustness_table(leverage_returns)


window_results.to_csv("../results/robustness_window.csv", index=False)
vol_window_results.to_csv("../results/robustness_vol_window.csv", index=False)
step_results.to_csv("../results/robustness_step.csv", index=False)
vol_target_results.to_csv("../results/robustness_vol_target.csv", index=False)
leverage_results.to_csv("../results/robustness_leverage.csv", index=False)